# Fundamentos de Ingeniería de Datos
## Ingesta y Fundamentos del ciclo de vida de datos


```
Fuente de datos→ CSV → Google Drive → SQLite (RAW)
```



### Configuración del entorno del proyecto

Antes de empezar, montamos Google Drive e importamos `utils.py`, el módulo que centraliza las rutas y funciones que usaremos en **todos los notebooks del curso**. Ejecuta estas celdas al inicio de cada sesión.


In [1]:
# ── Montar Google Drive ────────────────────────────────────────────────────

# from google.colab import drive
# drive.mount('/content/drive')


In [2]:
# ── Importar utils.py ──────────────────────────────────────────────────────

# utils.py contiene las rutas del proyecto y funciones compartidas. Debe
# estar en la raíz de tu carpeta en Drive o en una dirección en especifico:

#
#   CUGDL/Fundamentos_de_Ingenieria_de_Datos/Utilidades/utils.py
#


import sys

sys.path.append('/Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Utilidades')


from utils import (

    PROYECTO_BASE, DATASETS_RAW_PATH, DB_PATH, inspeccionar_dataset, listar_tablas
)

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {PROYECTO_BASE}')

print()



utils.py importado correctamente
Proyecto base: /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos



In [3]:
# ── Importar librerías y crear carpetas base ──────────────────────────────

import os
import json
import sqlite3
import pandas as pd

from datetime import datetime


os.makedirs(DATASETS_RAW_PATH, exist_ok=True)


# Carpeta para los entregables/documentación de cada unidad

DOCS_PATH = f'{PROYECTO_BASE}/Documentacion'

os.makedirs(DOCS_PATH, exist_ok=True)


print()
print('Entorno listo')
print()
print(f'  Datasets raw : {DATASETS_RAW_PATH}')
print(f'  Documentación: {DOCS_PATH}')
print()


Entorno listo

  Datasets raw : /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Datasets_raw
  Documentación: /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Documentacion



### PASO 1 — Configuración de la API de Kaggle

Para autenticarte con Kaggle necesitas tu **usuario** y tu **API key**. Puedes obtenerlos en [kaggle.com](https://www.kaggle.com) → Settings → API → **Create New Token**, lo que descarga un archivo `kaggle.json`.

Sube tu `kaggle.json` a Drive una sola vez y apunta la variable de entorno a esa ruta.


In [4]:
print(PROYECTO_BASE)

/Users/artdeco/Documents/CUGDL/Ingenieria_de_datos


In [5]:
# ── Opción recomendada: subir kaggle.json a Drive UNA sola vez y apuntar ahí ──
# 1. Descarga tu token en kaggle.com → Settings → API → Create New Token
# 2. Súbelo a: /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Credenciales/kaggle.json


import os

KAGGLE_JSON_PATH = f'{PROYECTO_BASE}/Credenciales/kaggle.json'

#/Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Credenciales/kaggle.json

if os.path.exists(KAGGLE_JSON_PATH):
    
    os.environ['KAGGLE_CONFIG_DIR'] = PROYECTO_BASE
    print()
    print('kaggle.json encontrado en Drive — usando esa configuración.')
    print()

else:

    # ── Alternativa: ingreso interactivo  ──

    from getpass import getpass
    os.environ['KAGGLE_USERNAME'] = input('Usuario de Kaggle: ')
    os.environ['KAGGLE_KEY']      = getpass('API key de Kaggle (no se mostrará en pantalla): ')
    print()
    print('Credenciales cargadas solo en esta sesión (no se guardan en el notebook).')
    print()




kaggle.json encontrado en Drive — usando esa configuración.



In [6]:
# Instalar la librería de Kaggle

!pip install kaggle --quiet
print()
print('----- Librería de Kaggle instalada -----')
print()



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip

----- Librería de Kaggle instalada -----



### PASO 2 — Explorar datasets disponibles en Kaggle

Antes de descargar, buscamos desde la API qué hay disponible sobre un tema de interés. Cambia `search='...'` por el campo que más te interese.


In [7]:
# Nos aseguramos de tener una versión reciente de la librería kaggle:
# versiones antiguas y nuevas de la API no siempre exponen los mismos atributos
# en los objetos de dataset (por eso el código de abajo usa getattr con opciones).


!pip install -q --upgrade kaggle
print()
print('----- Librería kaggle actualizada -----')
print()



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip

----- Librería kaggle actualizada -----



In [8]:
import kaggle


TEMA_BUSQUEDA = 'spotify 2022'   # cambia esto por tu campo de interés

print()
print(f'----- Buscando datasets sobre \"{TEMA_BUSQUEDA}\" en Kaggle -----\n')
print()

datasets = kaggle.api.dataset_list(search=TEMA_BUSQUEDA, sort_by='votes', file_type='csv')


for i, ds in enumerate(datasets[:10]):


    tamano = getattr(ds, 'size', None) or getattr(ds, 'totalBytes', None) or 'N/D'
    votos  = getattr(ds, 'voteCount', 'N/D')
    print(f'{i+1}. {ds.ref}')
    print(f'   {ds.title}  |  {tamano}  |  votos: {votos}')


Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token

----- Buscando datasets sobre "spotify 2022" en Kaggle -----


1. muhmores/spotify-top-100-songs-of-20152019
   Spotify Top 100 Songs of 2010-2019  |  N/D  |  votos: N/D
2. josephinelsy/spotify-top-hit-playlist-2010-2022
   Spotify Top Hit Playlist (2010-2023)  |  N/D  |  votos: N/D
3. byomokeshsenapati/spotify-song-attributes
   

### PASO 3 — Descarga de datasets

Vamos a descargar los datasets que usaremos en el curso. El primero (`raw_df1`) será el dataset **principal**, el que recorrerá las 4 capas del pipeline en los siguientes notebooks.

In [9]:
import os

# Crear carpeta de datos crudos (raw)

os.makedirs(DATASETS_RAW_PATH, exist_ok=True)

print(f'----- Carpeta lista: {DATASETS_RAW_PATH} -----')


----- Carpeta lista: /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Datasets_raw -----


In [10]:
# ── Datasets a descargar ────────────────────────────────────────────────────

# Coloca el identificador tal como aparece en Kaggle (usuario/nombre-dataset).
# El PRIMER elemento de la lista es tu dataset principal (df1): será el que
# uses en TODOS los notebooks siguientes del curso.

mis_datasets = [

    'alitaqishah/spotify-wrapped-2025-top-songs-and-artists', 
    'atharvasoundankar/spotify-global-streaming-data-2024', 
    'sveta151/spotify-top-chart-songs-2022'
    

    ]


In [11]:
# ── Descargar cada dataset ──────────────────────────────────────────────────

for ref in mis_datasets:
    nombre  = ref.replace('/', '_')
    carpeta = os.path.join(DATASETS_RAW_PATH, nombre)
    os.makedirs(carpeta, exist_ok=True)


    print()
    print(f'Descargando: {ref}')
    !kaggle datasets download -d {ref} -p {carpeta} --unzip --quiet
    print(f'   Listo → {carpeta}\n')
    print()



Descargando: alitaqishah/spotify-wrapped-2025-top-songs-and-artists
Dataset URL: https://www.kaggle.com/datasets/alitaqishah/spotify-wrapped-2025-top-songs-and-artists
License(s): CC0-1.0
   Listo → /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Datasets_raw/alitaqishah_spotify-wrapped-2025-top-songs-and-artists



Descargando: atharvasoundankar/spotify-global-streaming-data-2024
Dataset URL: https://www.kaggle.com/datasets/atharvasoundankar/spotify-global-streaming-data-2024
License(s): apache-2.0
   Listo → /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Datasets_raw/atharvasoundankar_spotify-global-streaming-data-2024



Descargando: sveta151/spotify-top-chart-songs-2022
Dataset URL: https://www.kaggle.com/datasets/sveta151/spotify-top-chart-songs-2022
License(s): copyright-authors
   Listo → /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Datasets_raw/sveta151_spotify-top-chart-songs-2022




In [12]:
# ── Cargar automáticamente el primer CSV de cada dataset descargado ────────

import glob

dataframes = {}

for i, ref in enumerate(mis_datasets, start=1):
    nombre_carpeta = ref.replace('/', '_')
    carpeta = os.path.join(DATASETS_RAW_PATH, nombre_carpeta)

    csvs = glob.glob(os.path.join(carpeta, '*.csv'))
    if not csvs:
        print(f'  Sin archivos .csv en {carpeta}, revisa la descarga.')
        continue

    clave = f'sp{i}'
    dataframes[clave] = pd.read_csv(csvs[0])
    print()
    print(f'{clave} ← {os.path.basename(csvs[0])}  ({dataframes[clave].shape[0]:,} filas, {dataframes[clave].shape[1]} columnas)')


sp = dataframes.get('sp')
print()



sp1 ← spotify_wrapped_2025_top50_artists.csv  (50 filas, 11 columnas)

sp2 ← Spotify_2024_Global_Streaming_Data.csv  (500 filas, 12 columnas)

sp3 ← spotify_top_charts_22.csv  (646 filas, 17 columnas)



### PASO 4 — Carga e inspección inicial de los datos (Preparación)

Con los datos ya en memoria, hacemos la **inspección inicial**: dimensiones, tipos de dato y nulos. Esto es la etapa de *preparación* del ciclo de vida — todavía no modificamos nada, solo **conocemos** el dataset. Usamos `inspeccionar_dataset()` de `utils.py`.


In [13]:
for clave, df in dataframes.items():

    inspeccionar_dataset(df, nombre=f'{clave} — {mis_datasets[int(clave[2:]) - 1]}')



  DATASET: sp1 — alitaqishah/spotify-wrapped-2025-top-songs-and-artists
  Filas    : 50
  Columnas : 11

  Tipos de datos y valores nulos:
    wrapped_2025_rank            int64        nulos: 0 (0.0%)
    artist_name                  str          nulos: 0 (0.0%)
    monthly_listeners_millions_mar2026 float64      nulos: 0 (0.0%)
    primary_genre                str          nulos: 0 (0.0%)
    country                      str          nulos: 0 (0.0%)
    followers_millions           int64        nulos: 0 (0.0%)
    grammy_wins                  int64        nulos: 0 (0.0%)
    debut_year                   int64        nulos: 0 (0.0%)
    gender                       str          nulos: 0 (0.0%)
    top_2025_song                str          nulos: 0 (0.0%)
    dataset_part                 str          nulos: 0 (0.0%)

  Primeras 3 filas:


,wrapped_2025_rank,artist_name,monthly_listeners_millions_mar2026,primary_genre,country,followers_millions,grammy_wins,debut_year,gender,top_2025_song,dataset_part
0,1,The Weeknd,110.4,R&B/Pop,Canada,112,4,2010,Male,Die With A Smile,Spotify Wrapped 2025 Top 50 Artists
1,2,Taylor Swift,107.5,Pop/Country,USA,110,14,2006,Female,Fortnight,Spotify Wrapped 2025 Top 50 Artists
2,3,Drake,88.6,Hip-Hop,Canada,95,4,2006,Male,Amen,Spotify Wrapped 2025 Top 50 Artists




  DATASET: sp2 — atharvasoundankar/spotify-global-streaming-data-2024
  Filas    : 500
  Columnas : 12

  Tipos de datos y valores nulos:
    Country                      str          nulos: 0 (0.0%)
    Artist                       str          nulos: 0 (0.0%)
    Album                        str          nulos: 0 (0.0%)
    Genre                        str          nulos: 0 (0.0%)
    Release Year                 int64        nulos: 0 (0.0%)
    Monthly Listeners (Millions) float64      nulos: 0 (0.0%)
    Total Streams (Millions)     float64      nulos: 0 (0.0%)
    Total Hours Streamed (Millions) float64      nulos: 0 (0.0%)
    Avg Stream Duration (Min)    float64      nulos: 0 (0.0%)
    Platform Type                str          nulos: 0 (0.0%)
    Streams Last 30 Days (Millions) float64      nulos: 0 (0.0%)
    Skip Rate (%)                float64      nulos: 0 (0.0%)

  Primeras 3 filas:


,Country,Artist,Album,Genre,Release Year,Monthly Listeners (Millions),Total Streams (Millions),Total Hours Streamed (Millions),Avg Stream Duration (Min),Platform Type,Streams Last 30 Days (Millions),Skip Rate (%)
0,Germany,Taylor Swift,1989 (Taylor's Version),K-pop,2019,23.10,3695.53,14240.35,4.28,Free,118.51,2.24
1,Brazil,The Weeknd,After Hours,R&B,2022,60.60,2828.16,11120.44,3.90,Premium,44.87,23.98
2,United States,Post Malone,Austin,Reggaeton,2023,42.84,1425.46,4177.49,4.03,Free,19.46,4.77




  DATASET: sp3 — sveta151/spotify-top-chart-songs-2022
  Filas    : 646
  Columnas : 17

  Tipos de datos y valores nulos:
    uri                          str          nulos: 0 (0.0%)
    artist_names                 str          nulos: 0 (0.0%)
    track_name                   str          nulos: 0 (0.0%)
    peak_rank                    int64        nulos: 0 (0.0%)
    weeks_on_chart               int64        nulos: 0 (0.0%)
    danceability                 float64      nulos: 0 (0.0%)
    energy                       float64      nulos: 0 (0.0%)
    key                          int64        nulos: 0 (0.0%)
    loudness                     float64      nulos: 0 (0.0%)
    mode                         int64        nulos: 0 (0.0%)
    speechiness                  float64      nulos: 0 (0.0%)
    acousticness                 float64      nulos: 0 (0.0%)
    instrumentalness             float64      nulos: 0 (0.0%)
    liveness                     float64      nulos: 0 (0.0%)
    tem

,uri,artist_names,track_name,peak_rank,weeks_on_chart,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,tempo,time_signature,duration_ms
0,spotify:track:02MWAaffLxlfxAUY7c5dvx,Glass Animals,Heat Waves,1,65,0.761,0.525,11,-6.900,1,0.0944,0.4400,0.000007,0.0921,80.870,4,238805
1,spotify:track:5PjdY0CKGZdEuoNab3yDmX,"The Kid LAROI, Justin Bieber",STAY (with Justin Bieber),1,37,0.591,0.764,1,-5.484,1,0.0483,0.0383,0.000000,0.1030,169.928,4,141806
2,spotify:track:3IAfUEeaXRX9s9UdKOJrFI,Anitta,Envolver,3,3,0.812,0.736,4,-5.421,0,0.0833,0.1520,0.002540,0.0914,91.993,4,193806


### PASO 5 — Almacenamiento en SQLite (capa RAW → Archivado)

Como buena práctica en ingeniería de datos, los datos ingeridos se guardan **sin modificaciones** en una capa raw. Esta es la etapa de *archivado*: los datos quedan persistidos de forma estable para que las siguientes capas del pipeline (EXTRACTED, TRANSFORMED, SERVING) puedan leer de aquí, siempre respetando la regla de oro:

> **Cada capa lee de la capa inmediatamente anterior. La capa RAW nunca se modifica.**


In [14]:
import sqlite3

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)


conn = sqlite3.connect(DB_PATH)

for clave, df in dataframes.items():
    tabla = f'raw_{clave}'
    df.to_sql(tabla, conn, if_exists='replace', index=False)
    print(f'{clave} → tabla \"{tabla}\" ({df.shape[0]:,} filas)')

conn.commit()
conn.close()

print()
print(f'\nBase de datos: {DB_PATH}')
print()


sp1 → tabla "raw_sp1" (50 filas)
sp2 → tabla "raw_sp2" (500 filas)
sp3 → tabla "raw_sp3" (646 filas)


Base de datos: /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Base_de_datos/base_datos.db



In [15]:
# ── Verificar las tablas creadas en la base de datos ───────────────────────
listar_tablas(DB_PATH)



  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  extracted_df1                       extracted_        16,598     15
  raw_df1                             raw_              16,598     11
  raw_sp1                             raw_                  50     11
  raw_sp2                             raw_                 500     12
  raw_sp3                             raw_                 646     17


['extracted_df1', 'raw_df1', 'raw_sp1', 'raw_sp2', 'raw_sp3']

### PASO 6 — Consumo: primera consulta sobre los datos ya archivados

Cerramos el ciclo de vida con un primer **consumo**: una consulta SQL directa sobre la capa RAW, tal como lo haría cualquier persona que ya tiene acceso a los datos ingeridos.


In [16]:
conn = sqlite3.connect(DB_PATH)


query = '''
    SELECT Name, COUNT(*) AS num_registros

    FROM raw_df1
    WHERE Name IS NOT NULL
    GROUP BY Name

'''

resultado = pd.read_sql_query(query, conn)
conn.close()


print('Consulta de ejemplo — conteo de registros por tipo:\n')
display(resultado)


Consulta de ejemplo — conteo de registros por tipo:



,Name,num_registros
0,'98 Koshien,1
1,.hack//G.U. Vol.1//Rebirth,1
2,.hack//G.U. Vol.2//Reminisce,1
3,.hack//G.U. Vol.2//Reminisce (jp sales),1
4,.hack//G.U. Vol.3//Redemption,1
...,...,...
11488,thinkSMART: Chess for Kids,1
11489,uDraw Studio,1
11490,uDraw Studio: Instant Artist,2
11491,wwe Smackdown vs. Raw 2006,1
